In [1]:
import os

In [2]:
os.chdir("../")

In [3]:
%pwd

'c:\\Users\\LENOVO\\Desktop\\end-to-end ML\\Wine_Quality_Prediction'

In [31]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    RandomForest: dict  # Updated for clarity
    metric_file_path: Path  # Matches config.yaml
    target_column: str


In [5]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories, save_json

In [35]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params["RandomForest"]
        schema =  self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path = config.model_path,
            RandomForest=params,
            metric_file_path=Path(config["metric_file_path"]),
            target_column = schema.name
           
        )

        return model_evaluation_config

In [36]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import numpy as np
import joblib

In [37]:
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from pathlib import Path
import json

class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2

    def save_results(self):
        # Load test data and trained model
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        # Prepare test features and target
        test_x = test_data.drop(columns=[self.config.target_column])
        test_y = test_data[self.config.target_column]

        # Make predictions
        predicted_qualities = model.predict(test_x)

        # Calculate evaluation metrics
        rmse, mae, r2 = self.eval_metrics(test_y, predicted_qualities)

        # Save results as JSON
        scores = {"rmse": rmse, "mae": mae, "r2": r2}
        with open(self.config.metric_file_path, "w") as f:  # ✅ Fixed attribute name
            json.dump(scores, f, indent=4)

        print("Evaluation metrics saved successfully.")


In [39]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.save_results()
except Exception as e:
    raise e

[2025-04-04 17:51:38,310: INFO: common: yaml file: config\config.yaml loaded successfully]


[2025-04-04 17:51:38,381: INFO: common: yaml file: params.yaml loaded successfully]
[2025-04-04 17:51:38,407: INFO: common: yaml file: schema.yaml loaded successfully]
[2025-04-04 17:51:38,417: INFO: common: created directory at: artifacts]
[2025-04-04 17:51:38,422: INFO: common: created directory at: artifacts/model_evaluation]
Evaluation metrics saved successfully.
